# Nemotron **v31** — stratified, balanced, eval-gated refine of the 0.85 adapter

## Why your runs did what they did
- The "84 with 8 steps + high loss" run **warm-started the 0.85 adapter** → 8 steps barely moved 888M
  params → the 84 is the **inherited 0.85**, not earned in training. Loss looks high because it's on
  the new data the model never fit — but the model is already good.
- The "67" full run **drifted off the 0.85 manifold**: `merged_tong_andyhard.csv` is wildly imbalanced
  (cipher 3102 vs cryptarithm_guess 158) and mixed-style → unstratified full SFT lets cipher swamp
  every batch and overwrites the good weights. **More training = worse, with this recipe.**

## What this notebook changes
1. **Balance** — cap each type at `CAP_PER_TYPE` (cipher 3102→800); rare types kept whole.
2. **Stratified batches** — interleave categories so no batch is 50 cipher in a row.
3. **Low LR 1e-5** — warm-start *refine*, not smash (5e-5 full run is what drifted to 0.67).
4. **Eval-gate → save the BEST checkpoint** — per-category greedy accuracy on a held-out slice every
   N steps; keep the best step, not the last (since more steps can hurt). This is the safety net the
   67 run lacked.

## Honest expectation
SFT is imitation → its ceiling is ≈ the teacher data quality (~0.85). This recipe's job is to **not
regress** while nudging the weak categories (cryptarithm/equation_guess) — best case ~0.85–0.87. To
truly pass 0.85 you need **RL** (the v30/v30b RAFT pipeline). Use this to safely fold the hard-cat
data in without losing 0.85.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096
USE_FLASH_ATTN = 1
SEED = 42

def _find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    return ""
SFT_DATA_PATH = _find("/kaggle/input/**/balanced_sft.csv",
                      r"F:/Hackathons/Kaggle-Nemotron/data_manipulation/balanced_sft.csv")
WARM_START_ADAPTER_DIR = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
BEST_DIR = os.path.join(OUT_DIR, "v31_best_adapter")

# ── balance + stratify ──
CAP_PER_TYPE = 800        # downsample majority types to this; rare types kept whole
STRATIFIED = True         # interleave categories across batches (no cipher-only batches)
HOLDOUT_PER_TYPE = 8      # held-out per category (NOT trained); pool for the eval-gate
EVAL_N_PER_TYPE = 4       # how many of the holdout to actually generate on each eval

# ── warm-start refine (gentle; 5e-5 full run drifted 0.85 -> 0.67) ──
ATTN_RANK = 32
LORA_ALPHA = 64
LEARNING_RATE = 1e-5      # LOW = refine
LR_SCHED = "cosine"
WARMUP_RATIO = 0.05
PER_DEV_BATCH = 1
GRAD_ACCUM = 16
NUM_EPOCHS = 1
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0

# ── eval-gate: keep the BEST checkpoint, not the last ──
EVAL_EVERY = 0            # optimizer steps between holdout evals; 0 -> auto (~4 over the run)
EVAL_MAX_NEW = 1024
SAVE_BEST = True

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
SMOKE = 1
print({"data": bool(SFT_DATA_PATH), "CAP_PER_TYPE": CAP_PER_TYPE, "stratified": STRATIFIED,
       "LR": LEARNING_RATE, "batch": PER_DEV_BATCH * GRAD_ACCUM, "SMOKE": SMOKE})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    _attn = "flash_attention_2" if USE_FLASH_ATTN else "eager"

    def _load(attn):
        return FastLanguageModel.from_pretrained(
            model_name=MODEL_PATH,
            max_seq_length=MODEL_MAX_LEN,
            load_in_4bit=False, load_in_8bit=False,
            full_finetuning=False,
            trust_remote_code=True,
            unsloth_force_compile=False,
            attn_implementation=attn,
            dtype=torch.bfloat16,
        )

    try:
        model, tokenizer = _load(_attn)
        print(f"Loaded with attn_implementation={_attn!r}")
    except Exception as e:
        print(f"[attn] {_attn} failed ({type(e).__name__}: {e}); falling back to eager")
        model, tokenizer = _load("eager")

    # Report the kernel actually in use. Per the forum (Benni): the native
    # modeling_nemotron_h.py loaded via trust_remote_code=True may leave FA2 OFF
    # even when requested -- the transformers-native impl (trust_remote_code=False)
    # is the one that enables FA2 + packed experts. Verify before trusting speed.
    try:
        _impl = getattr(model.config, "_attn_implementation", "?")
        print(f"[attn] effective config._attn_implementation = {_impl}")
    except Exception:
        pass

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"     # SFT loss wants right padding
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
# ── warm-start the 0.85 adapter as TRAINABLE (continue, don't reinit) ──
from unsloth import FastLanguageModel
from peft import PeftModel
import os, glob

def _resolve(d):
    if d and os.path.exists(os.path.join(d, "adapter_config.json")): return d
    h = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(h, key=len)[0]) if h else None
A = _resolve(WARM_START_ADAPTER_DIR)
assert A, "0.85 adapter not found -- set WARM_START_ADAPTER_DIR"
print("[warm-start] continuing 0.85 adapter from", A)
model = PeftModel.from_pretrained(model, A, is_trainable=True)
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try: model.config.use_cache = False
except Exception: pass
model.train()
model.print_trainable_parameters()


In [ ]:
# ── data: per-type cap (balance) + holdout split + assistant-masked tokens (types kept aligned) ──
import pandas as pd, re
from datasets import Dataset as HFDataset

SYSTEM_PROMPT = (
    "You solve deterministic logical-puzzle tasks. Infer the exact rule from the examples, apply it "
    "step by step, verify it reproduces the examples, then output the final answer once as "
    "\\boxed{...} with nothing after it.")

df = pd.read_csv(SFT_DATA_PATH).dropna(subset=["prompt", "answer"]).reset_index(drop=True)
TYPE = "type" if "type" in df.columns else None
if TYPE is None:
    df["type"] = "all"; TYPE = "type"

# carve a per-type HOLDOUT (never trained) + cap the majority types to balance
tr_parts, hd_parts = [], []
for t, grp in df.groupby(TYPE):
    grp = grp.sample(frac=1, random_state=SEED)
    hd_parts.append(grp.head(HOLDOUT_PER_TYPE))
    rest = grp.iloc[HOLDOUT_PER_TYPE:]
    if len(rest) > CAP_PER_TYPE:
        rest = rest.head(CAP_PER_TYPE)
    tr_parts.append(rest)
df_tr = pd.concat(tr_parts).reset_index(drop=True)
df_hd = pd.concat(hd_parts).reset_index(drop=True)
if SMOKE:
    df_tr = df_tr.groupby(TYPE, group_keys=False).head(8).reset_index(drop=True)
print("train per-type:", dict(df_tr[TYPE].value_counts()), "| total", len(df_tr))
print("holdout per-type:", dict(df_hd[TYPE].value_counts()))

PCOL, ACOL = "prompt", "answer"
CCOL = "generated_cot" if "generated_cot" in df.columns else ("cot" if "cot" in df.columns else None)

def _strip_boxed(t):
    tok, out, i = "\\boxed{", [], 0
    while i < len(t):
        j = t.find(tok, i)
        if j < 0: out.append(t[i:]); break
        out.append(t[i:j]); k = j + len(tok); d = 1
        while k < len(t) and d > 0:
            d += t[k] == "{"; d -= t[k] == "}"; k += 1
        i = k
    return "".join(out)

def _build(row):
    ans = str(row[ACOL]).strip()
    cot = str(row.get(CCOL, "") or "").replace("<think>", "").replace("</think>", "").strip()
    cot = re.sub(r"\n{3,}", "\n\n", _strip_boxed(cot)).strip() or "Work through it step by step."
    return f"<think>\n{cot}\n</think>\n\\boxed{{{ans}}}"

recs, record_types = [], []
for _, r in df_tr.iterrows():
    recs.append({"system": SYSTEM_PROMPT, "user": str(r[PCOL]) + PROMPT_SUFFIX, "assistant": _build(r)})
    record_types.append(str(r[TYPE]))

def _tok_mask(ex):
    full = [{"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]}]
    def render(m, g):
        try: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g, enable_thinking=True)
        except TypeError: return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=g)
    fid = tokenizer(render(full, False), add_special_tokens=False, truncation=True, max_length=TRAIN_MAX_LEN)["input_ids"]
    pid = tokenizer(render(full[:2], True), add_special_tokens=False)["input_ids"]
    lab = list(fid)
    for i in range(min(len(pid), len(fid))): lab[i] = -100
    return {"input_ids": fid, "labels": lab}

_ds0 = HFDataset.from_list(recs)
_toks = _ds0.map(_tok_mask, remove_columns=_ds0.column_names, desc="tokenize+mask")
_keep, _ktypes = [], []
for i, ex in enumerate(_toks):
    if any(l != -100 for l in ex["labels"]):
        _keep.append(ex); _ktypes.append(record_types[i])
train_ds = HFDataset.from_list(_keep)
record_types = _ktypes
print("train rows (well-formed):", len(train_ds), "| types:", len(set(record_types)))


In [ ]:
# ── stratified, masked-gather-CE SFT + per-category eval-gate that saves the BEST checkpoint ──
import os, time, gc, math, random, json, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Sampler
from transformers import (Trainer, TrainingArguments, TrainerCallback,
                          StoppingCriteria, StoppingCriteriaList)
os.environ["TORCHDYNAMO_DISABLE"] = "1"; os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

class Collate:
    def __init__(self, tok): self.pad = tok.pad_token_id
    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats); ii = []; lb = []; am = []
        for f in feats:
            ids = list(f["input_ids"]); la = list(f["labels"]); p = m - len(ids)
            ii.append(ids + [self.pad] * p); lb.append(la + [-100] * p); am.append([1] * len(ids) + [0] * p)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(lb)}
collator = Collate(tokenizer)

def build_stratified_index_order(labels, batch_size, seed):
    from collections import defaultdict
    by = defaultdict(list)
    for idx, lab in enumerate(labels): by[lab].append(idx)
    rng = random.Random(seed)
    for v in by.values(): rng.shuffle(v)
    nb = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(nb)]; order = list(range(nb)); rng.shuffle(order); a = 0
    for lab in sorted(by.keys()):
        for idx in by[lab]:
            batches[order[a % nb]].append(idx); a += 1
    return [i for b in batches for i in b]

class _OrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

# ── eval verifier ──
def extract_boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1:
        m = re.findall(r"\\boxed\{([^{}]*)\}", t); return m[-1].strip() if m else None
    d, j = 1, i + 7
    while j < len(t) and d > 0:
        if t[j] == "{": d += 1
        elif t[j] == "}": d -= 1
        j += 1
    return t[i + 7:j - 1].strip()
def compare_answer(s, p):
    if p is None: return False
    s, p = str(s).strip(), str(p).strip()
    if re.fullmatch(r"[01]+", s): return p.lower() == s.lower()
    try: return math.isclose(float(s), float(p), rel_tol=1e-2, abs_tol=1e-5)
    except Exception: return p.lower() == s.lower()
class StopOnBoxed(StoppingCriteria):
    def __init__(self, tok, plen): self.tok = tok; self.plen = plen; self.step = 0
    def __call__(self, ids, scores, **kw):
        self.step += 1
        if self.step % 8 != 0: return False
        s = self.tok.decode(ids[0][max(self.plen, ids.shape[1]-80):], skip_special_tokens=True)
        j = s.rfind("\\boxed{")
        if j < 0: return False
        d, k = 1, j + 7
        while k < len(s) and d > 0:
            if s[k] == "{": d += 1
            elif s[k] == "}": d -= 1
            k += 1
        return d == 0

class EvalBest(TrainerCallback):
    def __init__(self): self.best = -1.0; self.best_step = -1
    def _eval(self, step):
        from collections import Counter
        model.eval()
        try: model.config.use_cache = True
        except Exception: pass
        samp = df_hd.groupby(TYPE, group_keys=False).head(EVAL_N_PER_TYPE)
        hit, tot = Counter(), Counter()
        for _, r in samp.iterrows():
            msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": str(r["prompt"]) + PROMPT_SUFFIX}]
            try: txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
            except TypeError: txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            enc = tokenizer(txt, return_tensors="pt", truncation=True, max_length=MODEL_MAX_LEN - EVAL_MAX_NEW).to(model.device)
            plen = enc["input_ids"].shape[1]
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=EVAL_MAX_NEW, do_sample=False, temperature=None,
                                     top_p=None, top_k=None, pad_token_id=tokenizer.pad_token_id,
                                     stopping_criteria=StoppingCriteriaList([StopOnBoxed(tokenizer, plen)]))
            gen = tokenizer.decode(out[0][plen:], skip_special_tokens=True)
            ok = compare_answer(r["answer"], extract_boxed(gen))
            c = str(r[TYPE]); tot[c] += 1; hit[c] += int(ok)
            del out, enc
        torch.cuda.empty_cache()
        H, T = sum(hit.values()), sum(tot.values())
        acc = H / max(1, T)
        bycat = {c: f"{hit[c]}/{tot[c]}" for c in sorted(tot)}
        print(f"[eval step {step}] holdout acc={acc:.1%} ({H}/{T}) | {bycat}", flush=True)
        try: model.config.use_cache = False
        except Exception: pass
        model.train()
        if SAVE_BEST and acc > self.best:
            self.best = acc; self.best_step = step
            trainer.model.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
            print(f"   -> NEW BEST {acc:.1%}  saved to {BEST_DIR}", flush=True)
        return acc
    def on_step_end(self, args, state, control, **kw):
        if EVAL_EVERY and state.global_step > 0 and state.global_step % EVAL_EVERY == 0:
            self._eval(state.global_step)
        return control
    def on_train_end(self, args, state, control, **kw):
        self._eval(state.global_step)
        print(f"\nBEST holdout acc = {self.best:.1%} at step {self.best_step}. Best adapter -> {BEST_DIR}")
        return control

class STrainer(Trainer):
    def __init__(self, *a, strat_order=None, **k):
        super().__init__(*a, **k); self.strat_order = strat_order
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs); logits = out.logits
        sl = logits[:, :-1, :]; slb = labels[:, 1:].to(sl.device); mask = (slb != -100)
        V = sl.shape[-1]; fm = mask.reshape(-1)
        sel = sl.reshape(-1, V)[fm].float(); selab = slb.reshape(-1)[fm]
        loss = (logits.float().sum() * 0.0).requires_grad_(True) if selab.numel() == 0 else F.cross_entropy(sel, selab)
        return (loss, out) if return_outputs else loss
    def get_train_dataloader(self):
        if self.strat_order is None:
            return super().get_train_dataloader()
        return DataLoader(self.train_dataset, batch_size=self.args.per_device_train_batch_size,
                          sampler=_OrderSampler(self.strat_order), collate_fn=self.data_collator,
                          num_workers=self.args.dataloader_num_workers, pin_memory=self.args.dataloader_pin_memory,
                          drop_last=self.args.dataloader_drop_last)

import re
_eff = PER_DEV_BATCH * GRAD_ACCUM
_total = max(1, len(train_ds) // _eff * NUM_EPOCHS)
_warm = max(1, int(WARMUP_RATIO * _total))
_eval_every = EVAL_EVERY or max(1, _total // 4)
if SMOKE: _eval_every = 4
strat = build_stratified_index_order(record_types, _eff, SEED) if (STRATIFIED and len(set(record_types)) > 1) else None
print(f"steps~{_total} warmup={_warm} eval_every={_eval_every} stratified={strat is not None} LR={LEARNING_RATE}")

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "run"), num_train_epochs=NUM_EPOCHS,
    max_steps=8 if SMOKE else -1, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHED,
    warmup_steps=_warm, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM, optim="paged_adamw_8bit",
    bf16=True, gradient_checkpointing=False, remove_unused_columns=False, logging_steps=1,
    report_to="none", save_strategy="no", seed=SEED)
EVAL_EVERY = _eval_every

trainer = STrainer(model=model, args=args, train_dataset=train_ds, data_collator=collator,
                   strat_order=strat, callbacks=[EvalBest()])
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train()
print(f"train done {(time.time()-t0)/60:.1f} min | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")


In [ ]:
# ── package the BEST checkpoint (not the last) as submission.zip ──
import os, json, zipfile, shutil
SRC = BEST_DIR if os.path.exists(os.path.join(BEST_DIR, "adapter_config.json")) else os.path.join(OUT_DIR, "v31_final")
if SRC != BEST_DIR:
    os.makedirs(SRC, exist_ok=True)
    trainer.model.save_pretrained(SRC); tokenizer.save_pretrained(SRC)
    print("[save] no eval-best saved (eval never beat -1?) -> saved FINAL to", SRC)

cfgp = os.path.join(SRC, "adapter_config.json")
cfg = json.load(open(cfgp))
cfg["base_model_name_or_path"] = BASE_MODEL_NAME; cfg["inference_mode"] = True; cfg["lora_dropout"] = 0.0
json.dump(cfg, open(cfgp, "w"), indent=2)

_run = os.path.join(OUT_DIR, "run")
if os.path.isdir(_run): shutil.rmtree(_run, ignore_errors=True)

need = ["adapter_config.json", "adapter_model.safetensors"]
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUT_DIR
if all(os.path.exists(os.path.join(SRC, n)) for n in need):
    z = os.path.join(WORK, "submission.zip")
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for n in need:
            zf.write(os.path.join(SRC, n), n)
    print(f"submission.zip -> {z} ({os.path.getsize(z)/1e6:.0f} MB)  from {SRC}")
else:
    print("[save] missing adapter files in", SRC)
print("Eval-gate keeps the BEST step, not the last -> no silent drift like a full unstratified run.")
